In [1]:
import sys
if 'google.colab' in sys.modules:
    from IPython.core.getipython import get_ipython
    get_ipython().run_line_magic("pip", "install transformers sentencepiece accelerate")
    get_ipython().run_line_magic("pip", "install git+https://github.com/UlisseMini/activation_additions_hf")

In [7]:
import torch
import math
import activation_additions as aa

from typing import Dict, Union, Callable
from functools import partial
from transformers import AutoModelForCausalLM, AutoTokenizer
from activation_additions.compat import get_x_vector
from functools import lru_cache

from nltk.tokenize import PunktTokenizer

from numpy import array,polyfit

In [3]:
device: str = "mps" if torch.has_mps else "cuda" if torch.cuda.is_available() else "cpu"
_ = torch.set_grad_enabled(False)
device

C:\Users\itayb\AppData\Local\Temp\ipykernel_44928\1209224586.py:1: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  device: str = "mps" if torch.has_mps else "cuda" if torch.cuda.is_available() else "cpu"


'cuda'

In [4]:
MODEL = "openai-community/gpt2-xl"
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model.to_str_tokens = lambda t: [t.replace('Ġ', ' ') for t in tokenizer.tokenize(t)]
model.tokenizer = tokenizer
# In steering experimentation spaces were found to work well, this makes no sense and I hate it.
tokenizer.pad_token_id = int(model.tokenizer.encode(" ")[-1])

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 1,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
get_x_vector_preset: Callable = partial(
    get_x_vector,
    pad_method="tokens_right",
    model=model,
    custom_pad_id=tokenizer.pad_token_id,
)

In [6]:
@lru_cache(maxsize=1000)
def get_diff_vector(prompt_add: str, prompt_sub: str, layer: int):
    return aa.get_diff_vector(model, tokenizer, prompt_add, prompt_sub, layer)


@lru_cache
def run_aa(coeff: float, layer: int, prompt_add: str, prompt_sub: str, texts: tuple[str], loss_ignore_mod_tokens: bool = False):
    # todo: could compute act_diff for all layers at once, then a single fwd pass of cost for changing layer.
    act_diff = coeff * get_diff_vector(prompt_add, prompt_sub, layer)
    blocks = aa.get_blocks(model)
    with aa.pre_hooks([(blocks[layer], aa.get_hook_fn(act_diff))]):
        inputs = tokenizer(list(texts), return_tensors='pt', padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        output = model(**inputs,output_hidden_states=True, return_dict=True)

    logprobs = torch.log_softmax(output.logits.to(torch.float32), -1)
    token_loss = -logprobs[..., :-1, :].gather(dim=-1, index=inputs['input_ids'][..., 1:, None])[..., 0]
    return token_loss

In [ ]:
def calculate_perplexity_change(args_obj):
    sent_detector = PunktTokenizer()
    texts = args_obj['texts']
    loss_diff_in_bin = dict()
    element_count_in_bin = dict()
    for text in texts:
        sentences = tuple(sent_detector.tokenize(text.strip()))
        mean_text_log_loss = 0   
        token_loss = run_aa(args_obj["coeff"],args_obj["layer"],args_obj["prompt_add"],args_obj["prompt_sub"],sentences,args_obj["loss_ignore_mod_tokens"])
        abs_token_loss = run_aa(0., 0, ' ', ' ', texts=sentences) # cached
        mean_text_log_loss = sum(sent.mean() for sent in token_loss)/len(sentences)
        abs_mean_text_log_loss = sum(sent.mean() for sent in abs_token_loss)/len(sentences)

        rel_bin = texts[text]
        loss_diff_in_bin[rel_bin] = loss_diff_in_bin.get(rel_bin,0) + (mean_text_log_loss - abs_mean_text_log_loss)
        element_count_in_bin[rel_bin] = element_count_in_bin.get(rel_bin,0) + 1
    
    x = []
    y = []
    for bin in loss_diff_in_bin:
        perplexity_diff_in_bin = math.exp(-loss_diff_in_bin[bin])
        x.append(bin)
        y.append(perplexity_diff_in_bin.item())

    print(list(zip(x,y)))
    slope,_ = polyfit(x,y,1)
    return slope

In [204]:
def get_prompt_add(concept: str) -> str:
    return "love"
def get_prompt_sub(concept: str) -> str:
    return "hate"
def get_test_texts(concept: str) -> dict[str,float]:
    return {
        "I hate you because I love you\nI love you":0,
        "I hate you because you're so beautiful.": 0,
        "I hate you because I love you\nThe world is a stage":0,
        "I hate you because you are a girl.":2.8,
        "I hate you because you're not me.\nI hate you because I am me.":2.8,
        "I hate you because you are a man and I am a woman.":2.8
    }

In [205]:
def calculate_steering_efficacy_over_concept(concept):
    summand_dict = {
        "coeff":2,
        "layer":1,
        "prompt_add":get_prompt_add(concept),
        "prompt_sub":get_prompt_sub(concept),
        "texts": get_test_texts(concept),
        "loss_ignore_mod_tokens":False
    }
    return calculate_perplexity_change(summand_dict)
calculate_steering_efficacy_over_concept("weddings")

tensor(119.8988, device='cuda:0') tensor(4.7866, device='cuda:0')
tensor(232.3390, device='cuda:0') tensor(5.4482, device='cuda:0')
tensor(124.9963, device='cuda:0') tensor(4.8283, device='cuda:0')
tensor(186.7975, device='cuda:0') tensor(5.2300, device='cuda:0')
tensor(18.4630, device='cuda:0') tensor(2.9158, device='cuda:0')
tensor(36.2539, device='cuda:0') tensor(3.5905, device='cuda:0')
{0: tensor(477.2341, device='cuda:0'), 2.8: tensor(241.5145, device='cuda:0')} {0: tensor(387.3708, device='cuda:0'), 2.8: tensor(229.4296, device='cuda:0')} {0: 3, 2.8: 3}
[(0, -29.95440673828125), (2.8, -4.028297424316406)]


np.float64(9.259324754987443)